<a href="https://colab.research.google.com/github/YASHYOGESHAHIRE/02-value-investment-finance/blob/main/value_investment.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
 import pandas as pd
 import numpy as np
 import yfinance as yf
 import math

In [4]:
tickers = pd.read_csv('/content/top_50_indian_stocks.csv')
tickers.head()

,Ticker,Company Name
0,RELIANCE.NS,Reliance Industries
1,TCS.NS,Tata Consultancy Services
2,HDFCBANK.NS,HDFC Bank
3,INFY.NS,Infosys
4,ICICIBANK.NS,ICICI Bank


In [5]:
def fetch_values_of_stocks(tickers):
  value_cols=[
      "Ticker",
      "Price",
      "PE-ratio",
      "PB-ratio",
      "PS-ratio",
      "EV/EBITDA",
      "EV/GP"
  ]
  value_df=pd.DataFrame(columns=value_cols)
  for ticker in tickers:
    stock= yf.Ticker(ticker)

    price = np.nan # Default price to NaN
    hist_data = stock.history(period="1d")
    if not hist_data.empty and 'Close' in hist_data.columns and not hist_data['Close'].empty:
        price = hist_data['Close'].iloc[-1]
    else:
        print(f"Warning: No valid price data found for {ticker}. Assigning NaN.")

    # Initialize ratios to NaN
    pe_ratio, pb_ratio, ps_ratio, evEbitda, evGrossProfit = np.nan, np.nan, np.nan, np.nan, np.nan

    try:
        info = stock.info
        if info: # Ensure info dictionary is not empty
            pe_ratio = info.get("forwardPE", np.nan)
            pb_ratio = info.get("priceToBook", np.nan)
            ps_ratio = info.get("priceToSalesTrailing12months", np.nan)
            ev = info.get("enterpriseValue", np.nan)
            ebitda = info.get("ebitda", np.nan)

            if pd.notna(ev) and pd.notna(ebitda) and ebitda != 0:
                evEbitda = ev / ebitda

            grossMargins = info.get("grossMargins", np.nan)
            totalRevenue = info.get("totalRevenue", np.nan)

            grossProfit = np.nan
            if pd.notna(grossMargins) and pd.notna(totalRevenue):
                grossProfit = grossMargins * totalRevenue

            if pd.notna(ev) and pd.notna(grossProfit) and grossProfit != 0:
                evGrossProfit = ev / grossProfit
        else:
            print(f"Warning: No info data found for {ticker}. Assigning NaN to ratios.")

    except Exception as e:
        print(f"Error fetching info for {ticker}: {e}")

    value_df.loc[len(value_df)] = [ticker, price, pe_ratio, pb_ratio, ps_ratio, evEbitda, evGrossProfit]
  return value_df

In [6]:
ril=yf.Ticker("RELIANCE.NS")
ril.info


{'address1': 'Maker Chambers IV',
 'address2': '3rd Floor 222 Nariman Point',
 'city': 'Mumbai',
 'zip': '400021',
 'country': 'India',
 'phone': '91 22 3555 5000',
 'fax': '91 22 2204 2268',
 'website': 'https://www.ril.com',
 'industry': 'Oil & Gas Refining & Marketing',
 'industryKey': 'oil-gas-refining-marketing',
 'industryDisp': 'Oil & Gas Refining & Marketing',
 'sector': 'Energy',
 'sectorKey': 'energy',
 'sectorDisp': 'Energy',
 'longBusinessSummary': 'Reliance Industries Limited engages in hydrocarbon exploration and production, petroleum refining and marketing, petrochemicals, advanced materials and composites, renewable, financial services, retail, and digital services worldwide. It operates through the Oil to Chemicals, Oil and Gas, Retail, Digital Services, and Others segments. The company is involved in refining and marketing products, including liquefied petroleum gas, propylene, naphtha, gasoline, jet/aviation turbine fuel, kerosine oil, diesel, sulfur, and petroleum c

In [13]:
tickers_list=tickers['Ticker']
df=fetch_values_of_stocks(tickers_list)
df

ERROR:yfinance:$TATAMOTORS.NS: possibly delisted; no price data found  (period=1d) (Yahoo error = "No data found, symbol may be delisted")


ERROR:yfinance:HTTP Error 404: {"quoteSummary":{"result":null,"error":{"code":"Not Found","description":"Quote not found for symbol: TATAMOTORS.NS"}}}


,Ticker,Price,PE-ratio,PB-ratio,PS-ratio,EV/EBITDA,EV/GP
0,RELIANCE.NS,1291.000000,17.954712,1.932505,NaN,12.065035,5.613055
1,TCS.NS,2198.899902,13.291954,7.008960,NaN,10.856281,7.127766
2,HDFCBANK.NS,747.049988,11.888652,1.962265,NaN,NaN,NaN
3,INFY.NS,1197.500000,14.609312,5.198081,NaN,1093.840651,814.033092
4,ICICIBANK.NS,1262.099976,13.876246,2.489418,NaN,NaN,NaN
5,HINDUNILVR.NS,2121.500000,39.595154,10.227251,NaN,33.888950,15.124873
6,SBIN.NS,977.700012,9.529786,1.524414,NaN,NaN,NaN
7,BAJFINANCE.NS,889.400024,17.976105,4.850250,NaN,NaN,22.510702
8,BHARTIARTL.NS,1798.199951,21.551907,7.052594,NaN,10.440634,8.754300
9,ITC.NS,280.700012,15.856420,4.850862,NaN,12.106601,7.140017


In [14]:

df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 50 entries, 0 to 49
Data columns (total 7 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   Ticker     50 non-null     object 
 1   Price      49 non-null     float64
 2   PE-ratio   49 non-null     float64
 3   PB-ratio   49 non-null     float64
 4   PS-ratio   0 non-null      float64
 5   EV/EBITDA  40 non-null     float64
 6   EV/GP      42 non-null     float64
dtypes: float64(6), object(1)
memory usage: 3.1+ KB


In [15]:
numeric_cols = [
    "Price",
    "PE-ratio",
    "PB-ratio",
    "PS-ratio",
    "EV/EBITDA",
    "EV/GP"
]
for col in numeric_cols:
  df[col]=df[col].fillna(df[col].mean())
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 50 entries, 0 to 49
Data columns (total 7 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   Ticker     50 non-null     object 
 1   Price      50 non-null     float64
 2   PE-ratio   50 non-null     float64
 3   PB-ratio   50 non-null     float64
 4   PS-ratio   0 non-null      float64
 5   EV/EBITDA  50 non-null     float64
 6   EV/GP      50 non-null     float64
dtypes: float64(6), object(1)
memory usage: 3.1+ KB


In [10]:
from scipy import stats

In [16]:
percentile_metrics ={
    "PE-ratio" : "PE-Ratio_Percentile",
    "PB-ratio" : "PB-Ratio_Percentile",
    "PS-ratio" : "PS-Ratio_Percentile",
    "EV/EBITDA" : "EV/EBITDA_Percentile",
    "EV/GP" : "EV/GP_Percentile"
}
for metric,percentile in percentile_metrics.items():
  df[percentile]=df[metric].apply(lambda x: stats.percentileofscore(df[metric],x)/100)
df.head()

,Ticker,Price,PE-ratio,PB-ratio,PS-ratio,EV/EBITDA,EV/GP,PE-Ratio_Percentile,PB-Ratio_Percentile,PS-Ratio_Percentile,EV/EBITDA_Percentile,EV/GP_Percentile
0,RELIANCE.NS,1291.000000,17.954712,1.932505,NaN,12.065035,5.613055,0.46,0.16,NaN,0.28,0.20
1,TCS.NS,2198.899902,13.291954,7.008960,NaN,10.856281,7.127766,0.22,0.74,NaN,0.22,0.34
2,HDFCBANK.NS,747.049988,11.888652,1.962265,NaN,75.386413,45.309444,0.16,0.18,NaN,0.87,0.89
3,INFY.NS,1197.500000,14.609312,5.198081,NaN,1093.840651,814.033092,0.34,0.68,NaN,1.00,1.00
4,ICICIBANK.NS,1262.099976,13.876246,2.489418,NaN,75.386413,45.309444,0.28,0.28,NaN,0.87,0.89


In [18]:
from statistics import mean
df['Value Score'] = df[[value for value in percentile_metrics.values()]].mean(axis=1)
df

,Ticker,Price,PE-ratio,PB-ratio,PS-ratio,EV/EBITDA,EV/GP,PE-Ratio_Percentile,PB-Ratio_Percentile,PS-Ratio_Percentile,EV/EBITDA_Percentile,EV/GP_Percentile,Value Score
0,RELIANCE.NS,1291.000000,17.954712,1.932505,NaN,12.065035,5.613055,0.46,0.16,NaN,0.28,0.20,0.2750
1,TCS.NS,2198.899902,13.291954,7.008960,NaN,10.856281,7.127766,0.22,0.74,NaN,0.22,0.34,0.3800
2,HDFCBANK.NS,747.049988,11.888652,1.962265,NaN,75.386413,45.309444,0.16,0.18,NaN,0.87,0.89,0.5250
3,INFY.NS,1197.500000,14.609312,5.198081,NaN,1093.840651,814.033092,0.34,0.68,NaN,1.00,1.00,0.7550
4,ICICIBANK.NS,1262.099976,13.876246,2.489418,NaN,75.386413,45.309444,0.28,0.28,NaN,0.87,0.89,0.5800
5,HINDUNILVR.NS,2121.500000,39.595154,10.227251,NaN,33.888950,15.124873,0.84,0.84,NaN,0.60,0.56,0.7100
6,SBIN.NS,977.700012,9.529786,1.524414,NaN,75.386413,45.309444,0.10,0.10,NaN,0.87,0.89,0.4900
7,BAJFINANCE.NS,889.400024,17.976105,4.850250,NaN,75.386413,22.510702,0.48,0.58,NaN,0.87,0.68,0.6525
8,BHARTIARTL.NS,1798.199951,21.551907,7.052594,NaN,10.440634,8.754300,0.62,0.76,NaN,0.14,0.44,0.4900
9,ITC.NS,280.700012,15.856420,4.850862,NaN,12.106601,7.140017,0.38,0.60,NaN,0.30,0.36,0.4100


In [19]:
df=df.sort_values(by="Value Score",ascending=False)

In [20]:
df

,Ticker,Price,PE-ratio,PB-ratio,PS-ratio,EV/EBITDA,EV/GP,PE-Ratio_Percentile,PB-Ratio_Percentile,PS-Ratio_Percentile,EV/EBITDA_Percentile,EV/GP_Percentile,Value Score
48,DMART.NS,4144.200195,58.444714,11.023334,NaN,52.403024,26.232405,1.00,0.88,NaN,0.74,0.74,0.8400
26,TITAN.NS,4260.200195,49.756813,24.085938,NaN,48.435195,23.423066,0.96,0.98,NaN,0.70,0.70,0.8350
25,ADANIGREEN.NS,1525.699951,49.477158,13.027922,NaN,33.400840,30.351249,0.94,0.94,NaN,0.58,0.78,0.8100
27,DIVISLAB.NS,6623.000000,46.954030,10.482266,NaN,50.068627,26.640610,0.90,0.86,NaN,0.72,0.76,0.8100
47,PIDILITIND.NS,1476.599976,48.695374,13.877688,NaN,41.699376,18.089749,0.92,0.96,NaN,0.68,0.62,0.7950
23,TATAMOTORS.NS,2718.009177,23.592273,5.753018,NaN,75.386413,45.309444,0.68,0.70,NaN,0.87,0.89,0.7850
14,ASIANPAINT.NS,2686.699951,45.893230,12.048360,NaN,38.038899,16.346470,0.86,0.92,NaN,0.66,0.60,0.7600
37,BRITANNIA.NS,5120.500000,39.398956,24.154556,NaN,34.636099,15.309733,0.82,1.00,NaN,0.62,0.58,0.7550
3,INFY.NS,1197.500000,14.609312,5.198081,NaN,1093.840651,814.033092,0.34,0.68,NaN,1.00,1.00,0.7550
18,ULTRACEMCO.NS,10912.000000,24.926277,4.455898,NaN,75.386413,45.309444,0.70,0.54,NaN,0.87,0.89,0.7500


In [21]:
df.head()

,Ticker,Price,PE-ratio,PB-ratio,PS-ratio,EV/EBITDA,EV/GP,PE-Ratio_Percentile,PB-Ratio_Percentile,PS-Ratio_Percentile,EV/EBITDA_Percentile,EV/GP_Percentile,Value Score
48,DMART.NS,4144.200195,58.444714,11.023334,NaN,52.403024,26.232405,1.00,0.88,NaN,0.74,0.74,0.840
26,TITAN.NS,4260.200195,49.756813,24.085938,NaN,48.435195,23.423066,0.96,0.98,NaN,0.70,0.70,0.835
25,ADANIGREEN.NS,1525.699951,49.477158,13.027922,NaN,33.400840,30.351249,0.94,0.94,NaN,0.58,0.78,0.810
27,DIVISLAB.NS,6623.000000,46.954030,10.482266,NaN,50.068627,26.640610,0.90,0.86,NaN,0.72,0.76,0.810
47,PIDILITIND.NS,1476.599976,48.695374,13.877688,NaN,41.699376,18.089749,0.92,0.96,NaN,0.68,0.62,0.795
